In [ ]:
# delete cache files
import shutil, os

for folder in ["/content/bronze", "/content/silver", "/content/gold"]:
    if os.path.exists(folder): shutil.rmtree(folder)
print("deleted")

deleted


In [ ]:
# !pip install requests
# !pip install pyspark

In [ ]:
import requests
import json
import os
from datetime import datetime

In [ ]:
base_url = "https://ws.audioscrobbler.com/2.0"
api_key = "your_api_goes_here"
user = "your_user_goes_here"
limit = 50
format = "json"


In [ ]:
# bronze/raw layer
period = "1month"

endpoints = {
    "top_tracks": {
        "method": "user.getTopTracks",
        "user": user,
        "period": period,
        "limit": limit,
        "api_key": api_key,
        "format": format
    },
    "top_albums": {
        "method": "user.getTopAlbums",
        "user": user,
        "period": period,
        "limit": limit,
        "api_key": api_key,
        "format": format
    },
    "top_artists": {
        "method": "user.getTopArtists",
        "user": user,
        "period": period,
        "limit": limit,
        "api_key": api_key,
        "format": format
    }
}

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
# saves json file
for dataset_name, query_params in endpoints.items():
    try:
        response = requests.get(base_url, params=query_params)
        data = response.json()
        folder_path = f"/content/bronze/{dataset_name}"
        os.makedirs(folder_path, exist_ok=True)
        filename = f"{dataset_name}_{timestamp}.json"
        full_path = os.path.join(folder_path, filename)
        with open(full_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)

        print(f"{full_path} saved\n")

    except Exception as e:
        print(f"error {dataset_name}: {e}\n")


/content/bronze/top_tracks/top_tracks_20260607_053244.json saved

/content/bronze/top_albums/top_albums_20260607_053244.json saved

/content/bronze/top_artists/top_artists_20260607_053244.json saved



In [ ]:
# silver layer
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

spark = SparkSession.builder \
    .appName("lastfm_statistics_silver_layer") \
    .config("spark.sql.parquet.compression.codec", "snappy") \
    .getOrCreate()

# transformation/cleaning
os.makedirs("/content/silver", exist_ok=True)


# transformation/cleaning: top tracks
df_tracks_raw = spark.read.option("multiline", "true").json("/content/bronze/top_tracks/*.json")
df_tracks_exploded = df_tracks_raw.select(F.explode("toptracks.track").alias("track"))

df_tracks_silver = df_tracks_exploded.select(
    F.col("track.name").alias("track_name"),
    F.col("track.artist.name").alias("artist_name"),
    F.col("track.playcount").cast("int").alias("play_count"),
    F.col("track.`@attr`.rank").cast("int").alias("rank_position")
)

path_tracks = "/content/silver/top_tracks"
df_tracks_silver.write.mode("overwrite").parquet(path_tracks)


# transformation/cleaning: albums
df_albums_raw = spark.read.option("multiline", "true").json("/content/bronze/top_albums/*.json")
df_albums_exploded = df_albums_raw.select(F.explode("topalbums.album").alias("album"))

df_albums_silver = df_albums_exploded.select(
    F.col("album.name").alias("album_name"),
    F.col("album.artist.name").alias("artist_name"),
    F.col("album.playcount").cast("int").alias("play_count"),
    F.col("album.`@attr`.rank").cast("int").alias("rank_position")
)

path_albums = "/content/silver/top_albums"
df_albums_silver.write.mode("overwrite").parquet(path_albums)


# transformation/cleaning: artists
df_artists_raw = spark.read.option("multiline", "true").json("/content/bronze/top_artists/*.json")
df_artists_exploded = df_artists_raw.select(F.explode("topartists.artist").alias("artist"))

df_artists_silver = df_artists_exploded.select(
    F.col("artist.name").alias("artist_name"),
    F.col("artist.playcount").cast("int").alias("play_count"),
    F.col("artist.`@attr`.rank").cast("int").alias("rank_position")
)

path_artists = "/content/silver/top_artists"
df_artists_silver.write.mode("overwrite").parquet(path_artists)

print("saved at /content/silver/")

# df_tracks_silver.show(5, truncate=False)
# df_albums_silver.show(5, truncate=False)
# df_artists_silver.show(5, truncate=False)

saved at /content/silver/


In [ ]:
# gold layer
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import os

spark = SparkSession.builder \
    .appName("lastfm_statistics_gold_layer") \
    .getOrCreate()

os.makedirs("/content/gold", exist_ok=True)

df_tracks = spark.read.parquet("/content/silver/top_tracks")
df_albums = spark.read.parquet("/content/silver/top_albums")
df_artists = spark.read.parquet("/content/silver/top_artists")

# chart 1a: artists
df_certification_artists = df_artists.withColumn(
    "certification",
    F.when(F.col("play_count") >= 300, "Diamond")
     .when(F.col("play_count") >= 150, "Platinum")
     .when(F.col("play_count") >= 75, "Gold")
     .otherwise(None)
)

path_gold_artists = "/content/gold/certification_artists"
df_certification_artists.write.mode("overwrite").parquet(path_gold_artists)
df_certification_artists.select("rank_position", "artist_name", "play_count", "certification").show(10, truncate=False)


# chart 1b: tracks
window_spec_tracks = Window.orderBy(F.desc("play_count"))
df_certification_tracks = df_tracks.groupBy("track_name", "artist_name") \
    .agg(F.sum("play_count").alias("play_count")) \
    .withColumn("rank_position", F.row_number().over(window_spec_tracks)) \
    .withColumn(
        "medal",
        F.when(F.col("rank_position") == 1, "Gold")
         .when(F.col("rank_position") == 2, "Silver")
         .when(F.col("rank_position") == 3, "Bronze")
         .otherwise(None)
    ) \
    .select("rank_position", "medal", "track_name", "artist_name", "play_count")

path_gold_tracks = "/content/gold/certification_tracks"
df_certification_tracks.write.mode("overwrite").parquet(path_gold_tracks)

print(f"saved at {path_gold_tracks}")


# chart 1c: albums
df_certification_albums = df_albums.withColumn(
    "status",
    F.when(F.col("rank_position") <= 3, "Trending")
     .otherwise(None)
)

path_gold_albums_chart = "/content/gold/certification_albums"
df_certification_albums.write.mode("overwrite").parquet(path_gold_albums_chart)

print(f"album saved {path_gold_albums_chart}")
df_certification_albums.select("rank_position", "status", "album_name", "artist_name", "play_count").show(10, truncate=False)


# chart 2: album dominance
df_album_dominance = df_albums.join(
        df_artists.select("artist_name", F.col("play_count").alias("artist_total_plays")),
        on="artist_name",
        how="inner"
    ) \
    .withColumn("album_share_%", F.round((F.col("play_count") / F.col("artist_total_plays")) * 100, 2)) \
    .select(
        "artist_name",
        "album_name",
        F.col("play_count").alias("total_plays_album"),
        F.col("album_share_%").alias("unique_tracks_listened")
    ) \
    .orderBy(F.desc("total_plays_album"))

path_gold_dominance = "/content/gold/album_dominance"
df_album_dominance.write.mode("overwrite").parquet(path_gold_dominance)

print(f"\n saved at {path_gold_dominance}")
df_album_dominance.show(10, truncate=False)



# chart 3: fandom loyalty chart
df_loyalty = df_tracks.groupBy("artist_name") \
    .agg(
        F.sum("play_count").alias("total_plays"),
        F.countDistinct("track_name").alias("distinct_tracks"),
        F.round(F.sum("play_count") / F.countDistinct("track_name"), 2).alias("loop_index")
    ) \
    .filter("total_plays > 5 AND distinct_tracks >= 5") \
    .orderBy(F.desc("loop_index"))

path_gold_loyalty = "/content/gold/fandom_loyalty"
df_loyalty.write.mode("overwrite").parquet(path_gold_loyalty)

print(f"\nsaved at {path_gold_loyalty}")

# df_certification_tracks.show(10, truncate=False)
# df_album_dominance.show(5, truncate=False)
df_loyalty.show(5, truncate=False)

+-------------+---------------+----------+-------------+
|rank_position|artist_name    |play_count|certification|
+-------------+---------------+----------+-------------+
|1            |CHANGETHEWXRLD |327       |Diamond      |
|2            |Ally Minju     |295       |Platinum     |
|3            |ARTMS          |227       |Platinum     |
|4            |SHINee         |227       |Platinum     |
|5            |Som do Reino   |153       |Platinum     |
|7            |f(x)           |106       |Gold         |
|8            |VITOHRIA SOUNDS|80        |Gold         |
|9            |Ministério ZOE |77        |Gold         |
|10           |NMIXX          |77        |Gold         |
|11           |TAEMIN         |72        |NULL         |
+-------------+---------------+----------+-------------+
only showing top 10 rows
saved at /content/gold/certification_tracks
album saved /content/gold/certification_albums
+-------------+--------+------------------------------+--------------+----------+
|ran